## Load and Chunk

In [3]:
from langchain_community.document_loaders import PyMuPDFLoader
from  langchain_text_splitters import RecursiveCharacterTextSplitter

docs = PyMuPDFLoader('company_policy.pdf').load()
chunks = RecursiveCharacterTextSplitter(chunk_size=512, chunk_overlap=50).split_documents(docs)


## Embed and Index

In [4]:
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
db = Chroma.from_documents(chunks, OpenAIEmbeddings(), persist_directory='./chroma_db')

## Retreive

In [5]:
retriever = db.as_retriever(search_kwargs={'k': 5})
relevant_chunks = retriever.invoke('What is the remote work policy?')

## Generate with Context

In [7]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate.from_template(
    'Answer from context only. Context: {context}\nQuestion: {question}')
answer = (prompt | ChatOpenAI(model='gpt-4o')).invoke({
    'context': '\n'.join([c.page_content for c in relevant_chunks]),
    'question': 'What is the remote work policy?'})

print(answer)

content="The remote work policy at NexaTech Solutions Ltd outlines that remote working arrangements are available to all permanent employees who have successfully completed their 3-month probationary period. However, contractors, agency workers, and employees on fixed-term contracts of less than 6 months are not eligible unless an exception is granted by their Head of Department. Employees wishing to apply for remote work must complete a Remote Work Application Form (Form HR-22) and submit it to their line manager at least 10 working days before the intended start date. The applications are assessed based on criteria such as the nature of the role, the employee's performance record, team structure and business unit needs, home working environment suitability, and data security requirements. For a full remote arrangement of up to 5 days/week, VP-level approval is needed. Additionally, equipment and home office standards must be met." additional_kwargs={'refusal': None} response_metadata